## load/parse the original demarc XML file

In [21]:
from lxml import etree

# Load the XML file
tree = etree.parse('../data/elliottDiss.xml')
root = tree.getroot()
print(root.tag)  # Print the root element tag

{urn:oasis:names:tc:opendocument:xmlns:office:1.0}document-content


```
<text:h text:style-name="treInstance" text:outline-level="3"><text:bookmark-end
                    text:name="DOC44"/><text:bookmark-start text:name="INST97"/><text:span
                    text:style-name="T40">INST97: </text:span>A Demarcation of the Thracian
                    <text:span text:style-name="treLatin">peraea</text:span> of <text:span
                    text:style-name="trePlaceAncient">Thasos</text:span></text:h>
```

In [ ]:

description_template = {
    "type": "LinguisticObject",
    "classified_as": [
        {
            "id": "http://vocab.getty.edu/aat/300435416",
            "type": "Type",
            "_label": "Description",
            "classified_as": [
                {
                    "id": "http://vocab.getty.edu/aat/300418049",
                    "type": "Type",
                    "_label": "Brief Text",
                }
            ],
        }
    ],
    "content": "",
}    

# Find all text:h elements with text:style-name="treInstance"
namespaces = {'text': 'urn:oasis:names:tc:opendocument:xmlns:text:1.0', "demarc": "https://paregorios.org/demarc/"}
h_elements = root.findall(".//text:h[@text:style-name='treInstance']", namespaces)

# Extract text:name attribute from child text:bookmark-start elements
instances = dict()
for h in h_elements:
    bookmark_start = h.find(".//text:bookmark-start", namespaces)
    if bookmark_start is not None:
        instance_id = bookmark_start.get(f'{{{namespaces["text"]}}}name')
        if instance_id.startswith("INST"):
            num = instance_id[4:]  # Extract the number after "INST"
            if num.isdigit():
                try:
                    instances[instance_id]
                except KeyError:
                    # instances are Linked Art "concepts"
                    # https://linked.art/model/concept/#concept-schemes-and-sets
                    instances[instance_id] = {
                        "id": namespaces['demarc'] + instance_id,
                        "type": "Type", # the Linked Art catch-all, general concept class
                        "_label": (' '.join(''.join(h.itertext()).split())).split(":", 1)[-1].strip(),
                        
                    }
                    
                    description = description_template.copy()                    
                    # Find the next text:p element with text:style-name="treDisputeStatement"
                    dispute_statement = h.xpath("following::text:p[@text:style-name='treDisputeStatement'][1]", namespaces=namespaces)
                    if dispute_statement:
                        dispute_text = ' '.join(''.join(dispute_statement[0].itertext()).split())
                        if dispute_text:
                            description["content"] = dispute_text
                            instances[instance_id]["referred_to_by"] = [description]
                            continue
    raise RuntimeError("abject failure")
print(f"Extracted instance IDs: {len(instances)}")
from pprint import pprint
pprint(instances, indent=2)


Extracted instance IDs: 106
{ 'INST100': { '_label': 'Boundary Dispute Involving Ardea',
               'id': 'https://paregorios.org/demarc-ns/INST100',
               'referred_to_by': [ { 'classified_as': [ { '_label': 'Description',
                                                          'classified_as': [ { '_label': 'Brief '
                                                                                         'Text',
                                                                               'id': 'http://vocab.getty.edu/aat/300418049',
                                                                               'type': 'Type'}],
                                                          'id': 'http://vocab.getty.edu/aat/300435416',
                                                          'type': 'Type'}],
                                     'content': 'The corpus agrimensorum '
                                                'provides the only testimony '
            